In [1]:
#Import Julia packages
using DelimitedFiles
using FastaIO
using HTTP
using Random
using CodecZlib
using CSV
using DataFrames
using Revise
using PyCall
using Serialization

In [2]:
#Import Phyton modules
pd = pyimport_conda("pandas", "pandas")
random = pyimport_conda("random", "random")

PyObject <module 'random' from '/Users/lecce/.julia/conda/3/aarch64/lib/python3.12/random.py'>

#### Including functions needed to run the notebook

In [3]:
function correct_path(splitted_path, String)
    """Function to retrieve files in a given path.
    It takes as input:
        1) splitted path: the current path split by each separator
        2) String: the path to reach the wanted file 
    The function returns:
        1) the correct complete path to reach the file
    """
    S="" #String initialization
    for i in 1:(length(splitted_path)-1)
        #Constructing the path with the correct separator and removing the last term
        S=S*splitted_path[i]*"/" 
    end
    #Final Output
    return S*String 
end

correct_path (generic function with 1 method)

In [4]:
#Julia file 
includet(correct_path(split(pwd(), '/'), "Functions/Mapping GLORI functions.jl"));
#Phyton file 
push!(pyimport("sys")."path", correct_path(split(pwd(), '/'), "Functions"))
myscript = pyimport("TVT");

#### Import GRCh38 files 

> Transcripts annotation file

In [5]:
file_annotations_rna=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/109.20190905/GCF_000001405.39_GRCh38.p13/GCF_000001405.39_GRCh38.p13_rna.gbff.gz");
bytes=file_annotations_rna.body
io=GzipDecompressorStream(IOBuffer(bytes))
Annotations_rna=String(read(io));
gene_vec=Annotations_gene(Annotations_rna);

> Transcript sequences

In [6]:
file_transcripts=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/109.20190905/GCF_000001405.39_GRCh38.p13/GCF_000001405.39_GRCh38.p13_rna.fna.gz");
bytes=file_transcripts.body
decompressed=read(GzipDecompressorStream(IOBuffer(bytes)))
io=IOBuffer(decompressed)
reader=FastaReader(io);
human_transcripts=collect(reader);
long_transcripts,Dict_ind=longest_transcripts(human_transcripts,gene_vec);
indexes_long=sort(vcat(values(Dict_ind)...));

> Genomes annotations file

In [7]:
file_annotations_dna=HTTP.get("https://ftp.ncbi.nlm.nih.gov/refseq/H_sapiens/annotation/annotation_releases/109.20190905/GCF_000001405.39_GRCh38.p13/GCF_000001405.39_GRCh38.p13_genomic.gff.gz");
bytes=file_annotations_dna.body
io=GzipDecompressorStream(IOBuffer(bytes))
Annot=(String(read(io)));
Annotations_dna=Annotations_genome(Annot);

## GLORI HEK293T: Train, Validation, Test

 Importing first replicate dataset

In [8]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM7438377&format=file&file=GSM7438377%5FHEK293T%2DGLORI%2Drep1%2EFDR%2Ecsv%2Egz");
Dataset_m6A=CSV.read(file_http.body,DataFrame);

 Importing second replicate dataset

In [9]:
file_http2=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM7438378&format=file&file=GSM7438378%5FHEK293T%2DGLORI%2Drep2%2EFDR%2Ecsv%2Egz")
Dataset_m6A2=CSV.read(file_http2.body,DataFrame);

i) In order to have the same partition between train, validation and test as the ones used in the paper we open the file containing the gene names used:


In [10]:
 #Training transcripts list
gene_train=deserialize(pwd()*"/Data/gene_train.bin");
#Validation transcripts list
gene_valid=deserialize(pwd()*"/Data/gene_validation.bin"); 
#Splitting in training and validation datasets by pulling together replicate datasets (1) and (2). Test datasest built independenlty from replicate (1) or (2)
Train, Validate, Test1, Test2=split_train_val_test(Dataset_m6A, Dataset_m6A2; train=gene_train, validate=gene_valid);

ii) In order to have any random partition of datasets (as used for the first partition of data to create "gene_train.bin", "gene_validation.bin") it is possible to remove comments and run the code here below:

In [11]:
#For datasets with a generic random partitions different from the ones used to produce figures 
#Train, Validate, Test1, Test2=split_train_val_test(Dataset_m6A, Dataset_m6A2);
#serialize(pwd()*"/Data/gene_train.bin", unique!(Train[:,4]))
#serialize(pwd()*"/Data/gene_validation.bin", unique!(Validate[:,4]))

### Mapping

Mapping experimental m6As over mature transcripts for each dataset considered:

In [12]:
GENE_to_INFO_train=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Train,long_transcripts, 9,8;verb=true);
GENE_to_INFO_validate=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Validate,long_transcripts,9,8;verb=true);
GENE_to_INFO_test1=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Test1,long_transcripts,9,8;verb=false);
GENE_to_INFO_test2=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Test2,long_transcripts,9,8;verb=false);

## Additional Datasets

### HEK293T prior dataset

In [13]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432591&format=file&file=GSM6432591%5F293T%2DmRNA%2D2%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_prior=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,10,9;verb=false);

### HEK293T control conditions

In [14]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432590&format=file&file=GSM6432590%5F293T%2DmRNA%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HEK_CC=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,10,9;verb=false);

### HEK293T FTO

In [15]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432593&format=file&file=GSM6432593%5FFTO%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_FTO=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,10,9;verb=false);


### HEK293T STM2457

In [16]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432635&format=file&file=GSM6432635%5FGLORI%2D293T%2DSTM20%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HEK_STM=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,9,8;verb=false);

### HeLa Control Conditions

In [17]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432595&format=file&file=GSM6432595%5FHela%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HELA_CC=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,10,9;verb=false);

### HeLa Hypoxia

In [18]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432598&format=file&file=GSM6432598%5Fhypoxia%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HPX=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,10,9;verb=false);

### HeLa STM2457

In [19]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432631&format=file&file=GSM6432631%5FGLORI%2DHela%2DSTM20%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HELA_STM=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,9,8;verb=false);

### HeLa siMETTL3

In [20]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432607&format=file&file=GSM6432607%5FsiMETTL3%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HELA_siMETTL3=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,9,8;verb=false);

#### HeLa siMETTL14

In [21]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432611&format=file&file=GSM6432611%5FsiMETTL14%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HELA_siMETTL14=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,9,8;verb=false);

#### HeLa siYTHDF2

In [22]:
file_http=HTTP.get("https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSM6432623&format=file&file=GSM6432623%5FHela%2DsiYTHDF2%2D1%5F35bp%5Fm2%2Etotalm6A%2EFDR%2Ecsv%2Egz");
Dataset=CSV.read(file_http.body,DataFrame);
GENE_to_INFO_HELA_siYTHDF2=mapping_wrapped(human_transcripts,indexes_long,Annotations_dna,Annotations_rna,gene_vec,Dataset,long_transcripts,9,8;verb=false);

Saving all mapped datasets

In [23]:
serialize(pwd()*"/Data/Train_GLORI.bin", GENE_to_INFO_train)
serialize(pwd()*"/Data/Validation_GLORI.bin", GENE_to_INFO_validate)
serialize(pwd()*"/Data/Test1_GLORI.bin", GENE_to_INFO_test1)
serialize(pwd()*"/Data/Test2_GLORI.bin", GENE_to_INFO_test2)
serialize(pwd()*"/Data/Prior_GLORI.bin", GENE_to_INFO_prior)
serialize(pwd()*"/Data/HEK_CC_GLORI.bin", GENE_to_INFO_HEK_CC)
serialize(pwd()*"/Data/HEK_FTO_GLORI.bin", GENE_to_INFO_FTO)
serialize(pwd()*"/Data/HEK_STM2457_GLORI.bin", GENE_to_INFO_HEK_STM)
serialize(pwd()*"/Data/HELA_CC.bin", GENE_to_INFO_HELA_CC)
serialize(pwd()*"/Data/HELA_Hypoxia_GLORI.bin", GENE_to_INFO_HPX)
serialize(pwd()*"/Data/HELA_STM2457_GLORI.bin", GENE_to_INFO_HELA_STM)
serialize(pwd()*"/Data/HELA_siMETTL3_GLORI.bin", GENE_to_INFO_HELA_siMETTL3)
serialize(pwd()*"/Data/HELA_siMETTL14_GLORI.bin", GENE_to_INFO_HELA_siMETTL14)
serialize(pwd()*"/Data/HELA_siYTHDF2_GLORI.bin", GENE_to_INFO_HELA_siYTHDF2)
